# Parquet 数据读取耗时测试

读取后复权股票行情、行业指数行情和申万成分股权重，输出文件数、数据形状与总耗时。

总耗时包含文件发现、PyArrow 多线程读取和转换为 pandas DataFrame。重复运行可能受操作系统文件缓存影响。

In [1]:
from pathlib import Path
from time import perf_counter

import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq

DATA_ROOT = Path.home() / "Desktop/InternData/StockData/processed"
START_DATE = "2024-01-01"
END_DATE = "2024-01-31"
RESULTS = []

print(f"读取区间：{START_DATE} 至 {END_DATE}")

读取区间：2024-01-01 至 2024-01-31


In [2]:
def timed_read(name, root, date_column, columns, force_float=None):
    start_day = pd.Timestamp(START_DATE).date()
    end_day = pd.Timestamp(END_DATE).date()
    start_key = start_day.strftime("%Y%m%d")
    end_key = end_day.strftime("%Y%m%d")

    started = perf_counter()
    files = sorted(
        path
        for year in range(start_day.year, end_day.year + 1)
        for path in root.glob(f"year={year:04d}/month=*/*.parquet")
        if start_key <= path.stem <= end_key
    )

    if not files:
        seconds = perf_counter() - started
        RESULTS.append({"dataset": name, "files": 0, "rows": 0, "columns": 0, "seconds": seconds})
        print(f"{name}：未找到数据文件，耗时 {seconds:.4f} 秒")
        return pd.DataFrame(columns=columns)

    schema = pq.read_schema(files[0])
    if force_float:
        position = schema.get_field_index(force_float)
        schema = schema.set(position, pa.field(force_float, pa.float64()))

    dataset = ds.dataset([str(path) for path in files], format="parquet", schema=schema)
    date_filter = (
        (ds.field(date_column) >= pa.scalar(start_day, type=pa.date32()))
        & (ds.field(date_column) <= pa.scalar(end_day, type=pa.date32()))
    )
    table = dataset.to_table(columns=columns, filter=date_filter, use_threads=True)
    frame = table.to_pandas(types_mapper=pd.ArrowDtype)
    seconds = perf_counter() - started

    RESULTS.append({
        "dataset": name,
        "files": len(files),
        "rows": len(frame),
        "columns": len(frame.columns),
        "seconds": seconds,
    })
    print(f"{name}：{len(files)} 个文件，{len(frame):,} 行 × {len(frame.columns)} 列，耗时 {seconds:.4f} 秒")
    return frame

## 1. 后复权股票日行情

In [3]:
equity_daily = timed_read(
    name="后复权股票日行情",
    root=DATA_ROOT / "uqer_equity_daily_hfq",
    date_column="trade_date",
    columns=["trade_date", "symbol", "open", "high", "low", "close", "vwap", "amount", "is_open"],
)
equity_daily.head()

后复权股票日行情：22 个文件，109,998 行 × 9 列，耗时 0.0514 秒


,trade_date,symbol,open,high,low,close,vwap,amount,is_open
0,2024-01-02,000001.SZ,1258.915,1262.937,1234.782,1234.782,1245.068,1075742252.45,1
1,2024-01-02,000002.SZ,1734.407,1741.052,1686.229,1686.229,1701.575,830765500.05,1
2,2024-01-02,000004.SZ,110.027,112.35,109.685,110.3,110.773,46791154.0,1
3,2024-01-02,000005.SZ,10.612,11.005,10.612,11.005,10.862,6967071.0,1
4,2024-01-02,000006.SZ,332.929,334.383,323.479,324.933,326.523,117663239.29,1


## 2. 行业指数与宽基指数日行情

In [4]:
index_daily = timed_read(
    name="行业指数与宽基指数日行情",
    root=DATA_ROOT / "uqer_index_daily",
    date_column="trade_date",
    columns=["trade_date", "ticker", "index_family", "classification_version", "classification_name", "open", "high", "low", "close", "amount"],
    force_float="amount",
)
index_daily.head()

行业指数与宽基指数日行情：22 个文件，748 行 × 10 列，耗时 0.0129 秒


,trade_date,ticker,index_family,classification_version,classification_name,open,high,low,close,amount
0,2024-01-02,000300,broad_market,not_applicable,沪深300,3426.2684,3426.2684,3386.3522,3386.3522,184096097449.0
1,2024-01-02,000852,broad_market,not_applicable,中证1000,5891.1741,5891.2271,5854.5922,5854.7494,141351641579.0
2,2024-01-02,000905,broad_market,not_applicable,中证500,5431.4784,5439.4778,5409.4554,5409.4554,106839922689.0
3,2024-01-02,801010,sw_level1,SW2021,农林牧渔,2862.28,2891.51,2850.55,2876.23,9892680000.0
4,2024-01-02,801030,sw_level1,SW2021,基础化工,3459.73,3469.86,3448.9,3455.32,39863920000.0


## 3. 申万一级行业成分股权重

In [5]:
sw_weights = timed_read(
    name="申万一级行业成分股权重",
    root=DATA_ROOT / "uqer_sw_index_weights",
    date_column="effective_date",
    columns=["effective_date", "classification_version", "index_ticker", "classification_name", "constituent_symbol", "constituent_short_name", "weight"],
)
sw_weights.head()

申万一级行业成分股权重：未找到数据文件，耗时 0.0002 秒


,effective_date,classification_version,index_ticker,classification_name,constituent_symbol,constituent_short_name,weight


## 读取耗时汇总

In [6]:
timing_summary = pd.DataFrame(RESULTS)
timing_summary["seconds"] = timing_summary["seconds"].round(4)
timing_summary

,dataset,files,rows,columns,seconds
0,后复权股票日行情,22,109998,9,0.0514
1,行业指数与宽基指数日行情,22,748,10,0.0129
2,申万一级行业成分股权重,0,0,0,0.0002
